# Lab 10 — Positional embeddings for long context

A transformer looks at all tokens at once, so something has to tell it in which order they come.
In this lab you first see what happens when nothing does. You then build three answers from their formulas:
sinusoidal encodings, RoPE and ALiBi. Finally you plug them into a small GPT and run the experiment behind most of
Lecture 10: train it on short sequences, test it on long ones, and try to repair it.

Everything runs on one tiny model: a character-level GPT (4 layers, 4 heads,
128 dimensions, 0.8 M parameters) trained on Shakespeare with a context of **L = 64 characters**.
Its code, and the code that trains and evaluates it, are in this notebook (Exercise 5).

A cell with `# YOUR CODE HERE` is yours. The cell below it checks your code with a few `assert`s: if it runs without
an error, your code is right.

Stuck for more than five minutes? Ask for help.

## Setup

In [ ]:
import copy
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(min(4, torch.get_num_threads()))   # tiny tensors: more threads only add overhead

# TinyShakespeare, one token per character
text = open("data/input.txt").read()
chars = sorted(set(text))                                # the vocabulary: the 65 characters of the text
stoi = {c: i for i, c in enumerate(chars)}               # character -> id


def encode(s):
    return torch.tensor([stoi[c] for c in s])


def decode(ids):
    return "".join(chars[i] for i in ids)


data = encode(text)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]                # 90% to train, 10% to validate
L = 64                                                   # L_train: the context length of every model we train
print(f"{len(train_data):,} training characters, {len(val_data):,} validation characters, vocabulary of {len(chars)}")

## Exercise 1 — Self-attention ignores order

Self-attention computes, for the token at position $t$,

$$\text{out}_t=\sum_j \operatorname{softmax}_j\!\Big(\frac{q_t^\top k_j}{\sqrt d}\Big)\,v_j
\qquad\text{with}\qquad q_t=W_q x_t,\quad k_j=W_k x_j,\quad v_j=W_v x_j .$$

Nothing in this formula says where token $j$ is. A **causal** mask adds one restriction: token $t$ only attends to tokens $j\le t$.

**Your task.** Implement single-head attention with an optional causal mask.

In [ ]:
def single_head_attention(x, Wq, Wk, Wv, causal=False):
    """Single-head self-attention without any positional information.

    x           (T, d)  one row per token
    Wq, Wk, Wv  (d, d)  q_t = Wq x_t, k_t = Wk x_t, v_t = Wv x_t   (for all tokens at once: x @ Wq.T)
    causal              if True, token t only attends to tokens 0 .. t
    returns     (T, d)  row t is the sum over j of softmax_j(q_t . k_j / sqrt(d)) v_j
    """
    T, d = x.shape
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
x = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0]])                    # two tokens, d = 2
Wq, Wk = torch.eye(2), torch.eye(2)
Wv = torch.tensor([[1.0, 2.0],
                   [0.0, 1.0]])
assert torch.allclose(single_head_attention(x, Wq, Wk, Wv), torch.tensor([[1.3302, 0.3302],
                                                                          [1.6698, 0.6698]]), atol=1e-4)
assert torch.allclose(single_head_attention(x, Wq, Wk, Wv, causal=True), torch.tensor([[1.0000, 0.0000],
                                                                                       [1.6698, 0.6698]]), atol=1e-4)

**Three experiments.** The words below get random embeddings and there is no positional information anywhere.
Before you run each cell, predict what it prints.

- **(a)** No mask. "he loves she" against "she loves he".
- **(b)** Causal mask, one layer. "cat eats fish ?" against "fish eats cat ?": what happens to the output of the final "?"
- **(c)** Causal mask, two layers stacked. Same question.

In [ ]:
torch.manual_seed(0)
d = 16
E = {w: torch.randn(d) for w in ["he", "loves", "she", "cat", "eats", "fish", "?"]}   # one random vector per word
embed = lambda sentence: torch.stack([E[w] for w in sentence.split()])
W1 = [torch.randn(d, d) / math.sqrt(d) for _ in range(3)]     # Wq, Wk, Wv of a first layer
W2 = [torch.randn(d, d) / math.sqrt(d) for _ in range(3)]     # ... and of a second layer

def same(u, v):
    return "identical" if torch.allclose(u, v, atol=1e-5) else f"DIFFERENT (largest gap {(u - v).abs().max():.2f})"

a = single_head_attention(embed("he loves she"), *W1, causal=False)
b = single_head_attention(embed("she loves he"), *W1, causal=False)
print("(a) no mask")
print("    output of 'loves' in the two sentences:              ", same(a[1], b[1]))
print("    output of 'he' (first word in one, last in the other):", same(a[0], b[2]))

In [ ]:
s1, s2 = embed("cat eats fish ?"), embed("fish eats cat ?")

one_1 = single_head_attention(s1, *W1, causal=True)
one_2 = single_head_attention(s2, *W1, causal=True)
print("(b) causal, 1 layer:  output of '?':", same(one_1[-1], one_2[-1]))

two_1 = single_head_attention(one_1, *W2, causal=True)
two_2 = single_head_attention(one_2, *W2, causal=True)
print("(c) causal, 2 layers: output of '?':", same(two_1[-1], two_2[-1]))
print("    hint: after layer 1, the output of 'eats' is", same(one_1[1], one_2[1]))

**Question 1.** In (c) nothing tells the model about positions, and yet the output of "?" changes when "cat" and "fish"
swap places. Where does the order information come from? What does this say about a causal decoder that is trained
without any positional encoding (NoPE)?

*Your answer here*

## Exercise 2 — Sinusoidal positional encoding


The original Transformer adds a vector $p_k$ to the embedding of the token at position $k$:

$$p_k[2i]=\sin(k\,\theta_i),\qquad p_k[2i+1]=\cos(k\,\theta_i),\qquad \theta_i=10000^{-2i/d},\quad i=0,\dots,\tfrac d2-1 .$$

Each pair of dimensions $(2i,\,2i+1)$ is the hand of a clock that advances by $\theta_i$ radians per position.
Pair 0 is the seconds hand: a full turn every $2\pi\approx 6$ positions. The last pair is the hour hand: a full turn
takes almost $2\pi\cdot 10000$ positions.

In [ ]:
def sinusoidal_pe(n_pos, d, base=10000.0):
    """The table of sinusoidal positional encodings, shape (n_pos, d).

        PE[k, 2i]   = sin(k * theta_i)
        PE[k, 2i+1] = cos(k * theta_i)          theta_i = base^(-2i/d),   i = 0 .. d/2 - 1
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# d = 4: theta_0 = 1 and theta_1 = 10000^(-1/2) = 0.01
assert torch.allclose(sinusoidal_pe(3, 4), torch.tensor([[0.0000,  1.0000, 0.0000, 1.0000],     # position 0
                                                         [0.8415,  0.5403, 0.0100, 1.0000],     # position 1
                                                         [0.9093, -0.4161, 0.0200, 0.9998]]),   # position 2
                      atol=1e-4)

Each row of the picture is one $p_k$. Which dimensions change fastest when you go down the rows?

In [ ]:
plt.imshow(sinusoidal_pe(128, 64), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar()
plt.xlabel("dimension")
plt.ylabel("position k")
plt.title("Sinusoidal positional encoding")
plt.show()

## Exercise 3 — RoPE from scratch

RoPE adds nothing to the embeddings. Inside every attention layer it **rotates** the query of the token at position $m$
and the key of the token at position $n$. The pair of dimensions $(2i,\,2i+1)$ is rotated by the angle $m\,\theta_i$
(for the key: $n\,\theta_i$), with the same frequencies as before, $\theta_i=10000^{-2i/d}$, where $d$ is now the
dimension of one attention head:

$$x'[2i]=x[2i]\cos(m\theta_i)-x[2i+1]\sin(m\theta_i),\qquad x'[2i+1]=x[2i]\sin(m\theta_i)+x[2i+1]\cos(m\theta_i).$$

Values are not rotated.

### 3a. Frequencies and rotation

In [ ]:
def rope_theta(head_dim, base=10000.0):
    """The frequencies of RoPE, shape (head_dim/2,): theta_i = base^(-2i/head_dim) for the pair (2i, 2i+1)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert torch.allclose(rope_theta(8), torch.tensor([1.0, 0.1, 0.01, 0.001]))     # 10000^(-2i/8) = 10^(-i)

In [ ]:
def apply_rope(x, angles):
    """Rotate each pair (x[..., 2i], x[..., 2i+1]) of every token by its angle.

    x       (..., T, head_dim)
    angles  (T, head_dim/2)       angles[t, i] rotates the pair (2i, 2i+1) of token t
    returns a NEW tensor of the same shape as x. Do not modify x.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                  [1.0, 2.0, 3.0, 4.0]])          # two tokens, head_dim = 4
angles = torch.tensor([[0.0, 0.0],                # token 0: no rotation
                       [math.pi / 2, math.pi]])   # token 1: pair 0 by 90 degrees, pair 1 by 180 degrees
rotated = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                        [-2.0, 1.0, -3.0, -4.0]])
assert torch.allclose(apply_rope(x, angles), rotated, atol=1e-6)
assert torch.allclose(apply_rope(x[None, None], angles), rotated[None, None], atol=1e-6)  # with a batch and a head dimension
assert x.tolist() == [[1.0, 2.0, 3.0, 4.0], [1.0, 2.0, 3.0, 4.0]]                         # x itself is unchanged

### 3b. Only the distance matters

The score between a query at position $m$ and a key at position $n$ is the dot product of the two rotated vectors.

In [ ]:
torch.manual_seed(1)
head_dim = 32
theta = rope_theta(head_dim)
q, k = torch.randn(head_dim), torch.randn(head_dim)

def rope_score(q, k, m, n):
    """Score of the query q placed at position m and the key k placed at position n."""
    q_rotated = apply_rope(q[None], m * theta[None])     # q as a sequence of one token, rotated by the angles m theta_i
    k_rotated = apply_rope(k[None], n * theta[None])
    return (q_rotated * k_rotated).sum().item()

for m, n in [(5, 2), (105, 102), (1005, 1002), (5, 3)]:
    print(f"q at {m:4d}, k at {n:4d}:  score {rope_score(q, k, m, n):8.4f}")

### 3c. "Further apart, smaller attention?"

Does RoPE make far-away tokens score lower? The plot shows the score against the distance $m-n$
for a query and a key that are identical ($q=k$) and for unrelated random vectors (in gray: three more random pairs).

In [ ]:
distances = torch.arange(0, 129)

def scores_by_distance(q, k):
    """Scores of the query q at positions 0 .. 128 against the key k at position 0 (where it is not rotated)."""
    q_rotated = apply_rope(q.expand(len(distances), head_dim), distances[:, None] * theta[None])   # (129, head_dim)
    return q_rotated @ k

torch.manual_seed(1)
q, k = torch.randn(head_dim), torch.randn(head_dim)
for _ in range(3):
    plt.plot(distances, scores_by_distance(torch.randn(head_dim), torch.randn(head_dim)), color="lightgray")
plt.plot(distances, scores_by_distance(q, q), label="q = k")
plt.plot(distances, scores_by_distance(q, k), label="random q and k")
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("distance m - n between query and key")
plt.ylabel("score")
plt.title("RoPE: attention score against distance")
plt.legend()
plt.show()

**Question 3.**
1. Is "further apart, smaller score" true?
2. During generation new tokens keep arriving. Why can the keys of the old tokens stay in a cache untouched?
   (Lab 11 is built on this.)

*Your answer here*

## Exercise 4 — ALiBi

ALiBi adds nothing to the embeddings and rotates nothing. It adds a fixed penalty to the attention scores: for head
$h=1..H$, query $i$ and key $j\le i$, $\text{bias}_h[i,j]=-m_h\,(i-j)$ with $m_h=2^{-8h/H}$. The further back the key,
the larger the penalty, and every head has its own slope $m_h$.

In [ ]:
def alibi_bias(n_head, T):
    """ALiBi bias, shape (n_head, T, T): bias[h-1, i, j] = -m_h * (i - j) for j <= i, with m_h = 2^(-8h/n_head).
    The entries with j > i are masked by the causal mask anyway; set them to 0."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert torch.allclose(alibi_bias(2, 3), torch.tensor([[[ 0.0,      0.0,     0.0],      # head 1: m_1 = 2^(-4) = 1/16
                                                       [-1 / 16,   0.0,     0.0],
                                                       [-2 / 16,  -1 / 16,  0.0]],
                                                      [[ 0.0,      0.0,     0.0],      # head 2: m_2 = 2^(-8) = 1/256
                                                       [-1 / 256,  0.0,     0.0],
                                                       [-2 / 256, -1 / 256, 0.0]]]))

bias = alibi_bias(4, 512)
print("slopes m_h:", [round(-bias[h, 1, 0].item(), 4) for h in range(4)])
print("penalty of a key 63 positions back: ", [round(bias[h, 63, 0].item(), 2) for h in range(4)])
print("penalty of a key 500 positions back:", [round(bias[h, 500, 0].item(), 2) for h in range(4)])

## Exercise 5 — Train short, test long


### 5a. Plug your encodings into a GPT

Below is the whole model, a character-level GPT written like nanoGPT. `pos_mode` says how it learns about order:

| `pos_mode` | where the position enters |
|---|---|
| `none` | nowhere (NoPE). The causal mask is all the model has (Exercise 1) |
| `learned` | a trained vector per position, added to the token embedding (GPT-2, nanoGPT) |
| `sinusoidal` | the table of Exercise 2, added to the token embedding |
| `rope` | the rotation of Exercise 3, applied to q and k in every layer |
| `alibi` | the penalty of Exercise 4, added to the attention scores in every layer |

The attention in `Block.forward` is the one of Exercise 1, computed for 4 heads and a batch of windows at once:
`q`, `k` and `v` have the shape `(B, N_HEAD, T, HEAD_DIM)`.

**Your task.** Insert your functions where they belong: `sinusoidal_pe` in `GPT.forward`, `apply_rope` and
`alibi_bias` in `Block.forward`. For RoPE, the model keeps the frequencies $\theta_i$ in `theta`, and the angle of the
pair $i$ at position $m$ is $m\,\theta_i$.

In [ ]:
N_LAYER, N_HEAD, N_EMBD = 4, 4, 128
HEAD_DIM = N_EMBD // N_HEAD                    # 32: RoPE rotates 16 pairs of dimensions in every head
VOCAB_SIZE = len(chars)                        # 65

In [ ]:
class Block(nn.Module):
    """One layer of the transformer: causal self-attention, then an MLP, each added to its input."""

    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(N_EMBD, bias=False)
        self.qkv = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)     # queries, keys and values of all heads at once
        self.proj = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.ln_2 = nn.LayerNorm(N_EMBD, bias=False)
        self.mlp = nn.Sequential(nn.Linear(N_EMBD, 4 * N_EMBD, bias=False), nn.GELU(),
                                 nn.Linear(4 * N_EMBD, N_EMBD, bias=False))

    def forward(self, x, pos_mode, theta):
        B, T, _ = x.shape
        q, k, v = self.qkv(self.ln_1(x)).split(N_EMBD, dim=2)     # each (B, T, N_EMBD)
        q = q.view(B, T, N_HEAD, HEAD_DIM).transpose(1, 2)        # (B, N_HEAD, T, HEAD_DIM): one slice per head
        k = k.view(B, T, N_HEAD, HEAD_DIM).transpose(1, 2)
        v = v.view(B, T, N_HEAD, HEAD_DIM).transpose(1, 2)
        if pos_mode == "rope":
            # YOUR CODE HERE
            raise NotImplementedError()
        scores = q @ k.transpose(-2, -1) / math.sqrt(HEAD_DIM)     # (B, N_HEAD, T, T)
        if pos_mode == "alibi":
            # YOUR CODE HERE
            raise NotImplementedError()
        future = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        weights = F.softmax(scores.masked_fill(future, float("-inf")), dim=-1)
        y = (weights @ v).transpose(1, 2).reshape(B, T, N_EMBD)     # the outputs of the heads side by side
        x = x + self.proj(y)
        return x + self.mlp(self.ln_2(x))


class GPT(nn.Module):
    """A character-level GPT. pos_mode is "none", "learned", "sinusoidal", "rope" or "alibi"."""

    def __init__(self, pos_mode):
        super().__init__()
        self.pos_mode = pos_mode
        self.theta = rope_theta(HEAD_DIM)                          # the RoPE frequencies (used by pos_mode "rope")
        self.wte = nn.Embedding(VOCAB_SIZE, N_EMBD)                # one vector per character
        if pos_mode == "learned":
            self.wpe = nn.Embedding(L, N_EMBD)                     # one trained vector per position 0 .. L-1
        self.blocks = nn.ModuleList([Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD, bias=False)
        self.lm_head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)   # a score for every possible next character
        self.lm_head.weight = self.wte.weight                      # the same matrix as wte (weight tying, as in GPT-2)
        for p in self.parameters():                                # GPT-2 initialisation: small random matrices
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, idx):
        """idx: (B, T) character ids. Returns the logits (B, T, VOCAB_SIZE): position t predicts character t+1."""
        T = idx.size(1)
        x = self.wte(idx)                                          # (B, T, N_EMBD)
        if self.pos_mode == "learned":
            x = x + self.wpe(torch.arange(T))
        elif self.pos_mode == "sinusoidal":
            x = math.sqrt(N_EMBD) * x    # as in Vaswani et al.: the entries of p_k are of size 1, those of wte of size 0.02
            # YOUR CODE HERE
            raise NotImplementedError()
        for block in self.blocks:
            x = block(x, self.pos_mode, self.theta)
        return self.lm_head(self.ln_f(x))

In [ ]:
x = torch.randint(VOCAB_SIZE, (2, 10))                  # a batch of 2 windows of 10 random characters
for pos_mode in ["none", "learned", "sinusoidal", "rope", "alibi"]:
    assert GPT(pos_mode)(x).shape == (2, 10, VOCAB_SIZE)

### 5b. Train the models

`train` is nanoGPT's recipe for Shakespeare: 2000 steps on batches of 12 windows of L = 64 characters.
The cell after it trains the model five times, once per `pos_mode`, with the same initial seed and the same batches.
Each run takes from half a minute to a few minutes, depending on the machine.

**While it runs, read on and write `unseen_dims` in 5d.** You can edit cells while another one is running.

In [ ]:
def train(model, steps=2000, length=L, warmup=100):
    """Train the model on random windows of `length` characters of the training text, and return it.

    AdamW on batches of 12 windows. The learning rate grows linearly during `warmup` steps, then decays from 1e-3
    to 1e-4 along a cosine.
    """
    torch.manual_seed(0xB0B)                                              # the same batches in every run
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.99), weight_decay=0.1)
    for step in range(steps):
        if step < warmup:
            lr = 1e-3 * (step + 1) / (warmup + 1)
        else:
            lr = 1e-4 + 0.5 * (1e-3 - 1e-4) * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup)))
        for group in optimizer.param_groups:
            group["lr"] = lr

        starts = torch.randint(len(train_data) - length, (12,))
        x = torch.stack([train_data[s : s + length] for s in starts])          # (12, length)
        y = torch.stack([train_data[s + 1 : s + 1 + length] for s in starts])  # the next character at each position
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step % 500 == 0 or step == steps - 1:
            print(f"step {step:4d}: train loss {loss.item():.3f}")
    return model

In [ ]:
%%time
models = {}
for pos_mode in ["none", "learned", "sinusoidal", "rope", "alibi"]:
    print(f"pos_mode = {pos_mode}")
    torch.manual_seed(1337)                             # the same random initial weights for every model
    models[pos_mode] = train(GPT(pos_mode))

**Meet the model.** Here is what your RoPE model writes, one character at a time. We stop after 64 characters:
5c shows why.

In [ ]:
torch.manual_seed(0)
idx = encode("ROMEO:\n")[None]                                       # shape (1, 7): a batch with one sequence
with torch.no_grad():
    while idx.size(1) < L:
        probs = F.softmax(models["rope"](idx)[0, -1] / 0.8, dim=-1)   # the next character, at temperature 0.8
        idx = torch.cat([idx, torch.multinomial(probs, 1)[None]], dim=1)
print(decode(idx[0]))

### 5c. Beyond the training length

`loss_by_position` measures how well a model predicts the validation text at each position of windows of a given
length. Its mean over the positions is the number we compare. First within the training context, L = 64:

In [ ]:
@torch.no_grad()
def loss_by_position(model, length, n_tokens=25600):
    """Mean validation loss at each position 0 .. length-1 of windows of `length` characters, shape (length,).

    Entry t is the loss of predicting character t+1 of a window from its first t+1 characters. The n_tokens // length
    windows are drawn at random, but always the same ones: all the models are evaluated on the same text.
    """
    starts = torch.randint(len(val_data) - length, (n_tokens // length,), generator=torch.Generator().manual_seed(0))
    total = torch.zeros(length)
    for batch in starts.split(8192 // length):                       # 8192 characters at a time, to save memory
        x = torch.stack([val_data[s : s + length] for s in batch])
        y = torch.stack([val_data[s + 1 : s + 1 + length] for s in batch])
        total += F.cross_entropy(model(x).transpose(1, 2), y, reduction="none").sum(dim=0)
    return total / len(starts)


print("validation loss with the training context, L = 64 characters")
for name, model in models.items():
    print(f"  {name:11s} {loss_by_position(model, L).mean():.3f}")

All five were trained on windows of 64 characters. What happens with 128, 256, 512?
Start with the model that has a learned position table:

In [ ]:
window = val_data[: 2 * L][None]                 # 128 characters: twice the training context
try:
    models["learned"](window)
except IndexError as error:
    print("IndexError:", error)

Its table `wpe` has one row for each position 0 .. 63 and nothing else. Position 127 has no row, so the model cannot
even run: a learned table does not extrapolate beyond the training context.

**Your task.** Measure all five models at several lengths.

In [ ]:
def extrapolation_experiment(models, lengths):
    """Mean validation loss of each model at each evaluation length: {name: {length: loss}}.

    The loss of a model at one length is the mean over the positions of loss_by_position(model, length).
    A model that cannot run at some length raises an IndexError: store float("nan") for it.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
check = extrapolation_experiment({"none": models["none"], "learned": models["learned"]}, [L, 2 * L])
assert list(check) == ["none", "learned"] and list(check["none"]) == [L, 2 * L]
assert check["none"][L] == loss_by_position(models["none"], L).mean().item()
assert math.isnan(check["learned"][2 * L])

`show_results` prints results like these as a table (`nan`: the model cannot run) and plots them. We use it again
in Exercise 6.

In [ ]:
def show_results(results, title):
    """Print {name: {length: loss}} as a table with one row per name, and plot one line per name."""
    first_row = list(results.values())[0]                              # all the rows have the same lengths
    lengths = list(first_row)
    print(" " * 30 + "".join(f"{length:>8d}" for length in lengths))
    for name, losses in results.items():
        print(f"{name:30s}" + "".join(f"{losses[length]:8.3f}" for length in lengths))
        plt.plot(lengths, [losses[length] for length in lengths], marker="o", label=name)
    plt.axvline(L, color="gray", linestyle="--", linewidth=1)          # the training length
    plt.xscale("log", base=2)
    plt.xticks(lengths, lengths)
    plt.xlabel("evaluation length (characters)")
    plt.ylabel("mean validation loss")
    plt.title(title)
    plt.legend()
    plt.show()

Before you run the next cell, write down the ranking you expect at 8 L = 512 characters.

In [ ]:
%%time
results = extrapolation_experiment(models, [L, 2 * L, 4 * L, 8 * L])
show_results(results, "Validation loss against evaluation length")

### 5d. Where does RoPE break?

The curve below is the loss *at each position* of windows of 4 L = 256 characters: position 0 has one character of
context, position 255 has 256. ALiBi is shown for comparison.

In [ ]:
curves = {name: loss_by_position(models[name], 4 * L, n_tokens=200 * 4 * L) for name in ["rope", "alibi"]}  # 200 windows
for name, curve in curves.items():
    plt.plot(range(0, 4 * L, 16), curve.view(-1, 16).mean(dim=1), label=name)   # the mean of every 16 positions
plt.axvline(L, color="gray", linestyle="--", linewidth=1)                       # the training length
plt.xlabel("position in the window")
plt.ylabel("mean validation loss")
plt.title("Validation loss at each position of the window")
plt.legend()
plt.show()

print("mean loss in each quarter of the window:   [0, L)   [L, 2L)  [2L, 3L)  [3L, 4L)")
for name, curve in curves.items():
    print(f"  {name:38s}" + "".join(f"{curve[i * L:(i + 1) * L].mean():9.3f}" for i in range(4)))

The score of RoPE depends on the distance $m-n$ only, and in training that distance never exceeds $L-1$.
The pair $i$ turns by $\theta_i$ per position, so within the training context it sees the angles from 0 to
$\theta_i\,L$. If $\theta_i\,L<2\pi$, the pair never completes a full turn: part of the circle is never seen.

**Your task.** Return the pairs to which this happens.

In [ ]:
def unseen_dims(head_dim, L, base=10000.0):
    """Indices i of the RoPE pairs (2i, 2i+1) that never complete a full turn within L positions:
    theta_i * L < 2 pi, with theta_i = base^(-2i/head_dim). Returns a list of integers."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert unseen_dims(8, 64) == [2, 3]                  # theta * 64 = [64, 6.4, 0.64, 0.064], and 2 pi = 6.28
assert unseen_dims(HEAD_DIM, L) == list(range(5, 16))

In [ ]:
theta = rope_theta(HEAD_DIM)
unseen = unseen_dims(HEAD_DIM, L)

print(" pair i   wavelength 2pi/theta_i   turns within L = 64")
for i, t in enumerate(theta.tolist()):
    note = "   <- never completes a turn in training" if i in unseen else ""
    print(f"   {i:2d}   {2 * math.pi / t:14.1f}          {L * t / (2 * math.pi):8.2f}{note}")

**Question 5.** RoPE only looks at relative distances. Why does it still fail beyond L? Use your table of pairs and the
loss-by-position curve. Why is ALiBi flat, and why does `sinusoidal` collapse at once while `rope` degrades gradually?

*Your answer here*

## Exercise 6 — Extending RoPE

If the problem is angles that were never seen, slow the rotation down. Every method below replaces $\theta_i$ by
$\theta'_i=f(s,i)\,\theta_i$, where $s$ is how many times longer than $L$ we want to go.

| method | $\theta'_i$ | idea |
|---|---|---|
| **PI** | $\theta_i/s$ | divide every position by $s$: all of them fall back into the trained range |
| **NTK-aware** | $\big(\text{base}\cdot s^{\,d/(d-2)}\big)^{-2i/d}$, that is $f(s,i)=s^{-2i/(d-2)}$ | leave the fastest pair alone, divide the slowest by $s$, interpolate in between |
| **YaRN** | $(1-\gamma_i)\,\theta_i/s+\gamma_i\,\theta_i$ | divide only the pairs that turn too slowly, with a ramp $\gamma_i$ |
| **dynamic** | $s=\max(1,\ L_\text{current}/L)$ | choose $s$ from the length of the current sequence |

For YaRN, $r_i=L\,\theta_i/2\pi$ is the number of turns of pair $i$ within the training context (the column you printed
in 5d) and $\gamma_i=\operatorname{clamp}\big(\tfrac{r_i-\alpha}{\beta-\alpha},\,0,\,1\big)$: pairs with fewer than $\alpha$ turns
are fully interpolated, pairs with more than $\beta$ turns are left alone.

### 6a. The formulas

Each function returns the new frequencies $\theta'_i$, a tensor of shape `(head_dim/2,)`. Here $d$ is `head_dim`.

In [ ]:
def theta_pi(head_dim, s, base=10000.0):
    """Position Interpolation: every theta_i is divided by s (the same as dividing the positions by s)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert torch.allclose(theta_pi(8, 2), torch.tensor([0.5, 0.05, 0.005, 0.0005]))

In [ ]:
def theta_ntk(head_dim, s, base=10000.0):
    """NTK-aware scaling: the usual formula with the base replaced by base * s^(d/(d-2)), d = head_dim."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# the fastest pair is untouched, the slowest is divided by s = 2, the others lie in between
assert torch.allclose(theta_ntk(8, 2), torch.tensor([1.0, 0.1 / 2 ** (1 / 3), 0.01 / 2 ** (2 / 3), 0.001 / 2]))

In [ ]:
def dynamic_factor(seq_len, L_train):
    """Dynamic scaling: the factor s for a sequence of seq_len tokens, s = max(1, L_current / L_train)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert [dynamic_factor(n, 64) for n in [32, 64, 128, 256]] == [1, 1, 2, 4]

The scaling factor $\theta'_i/\theta_i$ of each pair, for $s=4$:

In [ ]:
theta = rope_theta(HEAD_DIM)
plt.plot(theta_pi(HEAD_DIM, 4) / theta, marker="o", label="PI")
plt.plot(theta_ntk(HEAD_DIM, 4) / theta, marker="o", label="NTK-aware")
plt.xlabel("pair i   (0 turns fastest, 15 slowest)")
plt.ylabel("scaling factor theta'_i / theta_i")
plt.title("How much each RoPE pair is slowed down (s = 4)")
plt.legend()
plt.show()

### 6b. Does it work without any training?

Your RoPE model reads its frequencies from `model.theta`. Replace them, and the model rotates q and k by the new
angles, with the weights it was trained with:

```python
model = copy.deepcopy(models["rope"])        # a copy: models["rope"] itself stays as it was trained
model.theta = theta_pi(HEAD_DIM, 2)
```

**Your task.** Evaluate one scaling method at several lengths, either with a fixed $s$ or with the dynamic $s$.

In [ ]:
def scaling_experiment(theta_fn, lengths, dynamic, s_static=4.0):
    """Validation loss of the trained RoPE model with scaled frequencies, at each length: {length: loss}.

    theta_fn  theta_pi, theta_ntk or theta_yarn: the model gets the frequencies theta_fn(HEAD_DIM, s)
    dynamic   True: s = dynamic_factor(length, L) for each length.  False: s = s_static at every length.
    The loss: loss_by_position(model, length).mean().item()
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert scaling_experiment(theta_pi, [L], dynamic=True) == {L: results["rope"][L]}    # s = 1 at L: nothing changes

In [ ]:
%%time
lengths = [L, 2 * L, 4 * L]
scaled = {
    "no scaling":              {length: results["rope"][length] for length in lengths},
    "PI, s = 4 always":        scaling_experiment(theta_pi, lengths, dynamic=False),
    "NTK-aware, s = 4 always": scaling_experiment(theta_ntk, lengths, dynamic=False),
    "PI, dynamic s":           scaling_experiment(theta_pi, lengths, dynamic=True),
    "NTK-aware, dynamic s":    scaling_experiment(theta_ntk, lengths, dynamic=True),
}
show_results(scaled, "RoPE scaling without any fine-tuning")

**Optional: YaRN.** The paper uses $\alpha=1$, $\beta=32$ for a model trained on 4096 tokens, where the fastest pairs
turn hundreds of times within the context. With L = 64 the fastest pair only turns 10 times, so $\beta=32$ would
interpolate every pair, which is PI again. We retuned the two numbers on the training split and use
**α = 0.25, β = 2** in this lab: they are the default values below. Skip the next three cells if you are short of
time: the rest of the exercise does not need them.

In [ ]:
def theta_yarn(head_dim, s, L_train=L, alpha=0.25, beta=2.0, base=10000.0):
    """YaRN: theta'_i = (1 - gamma_i) * theta_i / s + gamma_i * theta_i, with the ramp
    gamma_i = clamp((r_i - alpha) / (beta - alpha), 0, 1) and r_i = L_train * theta_i / (2 pi)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# r = 64 * [1, 0.1, 0.01, 0.001] / (2 pi) = [10.2, 1.02, 0.10, 0.01] turns, so gamma = [1, 0.44, 0, 0]
assert torch.allclose(theta_yarn(8, 2, L_train=64, alpha=0.25, beta=2.0), torch.tensor([1.0, 0.0720, 0.005, 0.0005]),
                      atol=1e-4)

In [ ]:
theta = rope_theta(HEAD_DIM)
plt.plot(theta_pi(HEAD_DIM, 4) / theta, marker="o", label="PI")
plt.plot(theta_ntk(HEAD_DIM, 4) / theta, marker="o", label="NTK-aware")
plt.plot(theta_yarn(HEAD_DIM, 4) / theta, marker="o", label="YaRN")
plt.xlabel("pair i   (0 turns fastest, 15 slowest)")
plt.ylabel("scaling factor theta'_i / theta_i")
plt.title("How much each RoPE pair is slowed down (s = 4)")
plt.legend()
plt.show()

scaled["YaRN, dynamic s"] = scaling_experiment(theta_yarn, lengths, dynamic=True)
show_results(scaled, "RoPE scaling without any fine-tuning")

### 6c. PI with a little fine-tuning

The PI paper fine-tunes the model with the squeezed positions. We do the same for 200 steps on windows of 2 L
(up to a minute on a slow machine).

In [ ]:
%%time
pi_model = copy.deepcopy(models["rope"])
pi_model.theta = theta_pi(HEAD_DIM, 2)
pi_finetuned = train(copy.deepcopy(pi_model), steps=200, length=2 * L, warmup=20)

after_finetuning = extrapolation_experiment({
    "no scaling, as trained":      models["rope"],
    "PI s = 2, as trained":        pi_model,
    "PI s = 2, fine-tuned at 2 L": pi_finetuned,
}, lengths)
show_results(after_finetuning, "PI, before and after fine-tuning")

A control: the same 200 steps on windows of 2 L, **without** any scaling.

In [ ]:
%%time
plain_finetuned = train(copy.deepcopy(models["rope"]), steps=200, length=2 * L, warmup=20)

after_finetuning.update(extrapolation_experiment({"no scaling, fine-tuned at 2 L": plain_finetuned}, lengths))
show_results(after_finetuning, "PI, before and after fine-tuning, and a control")

**Question 6.**
1. Why does PI hurt *short* sequences (row "PI, s = 4 always", column L), and how does dynamic scaling fix that?
2. Without fine-tuning, why is NTK-aware scaling so much better than PI? Look at your plot of the scaling factors.
3. After 200 steps the PI model is fine. Is that thanks to PI? Look at the control.

*Your answer here*

## Bonus

### B1. DroPE: drop the positional encoding after training

The idea of DroPE: a positional encoding helps the model to train, but it is also what breaks beyond L. So train with
RoPE, then remove it, and "recalibrate" the model for a few steps without any positional encoding.

**Your task.** Copy your RoPE model, set its `pos_mode` to `"none"` and fine-tune it for `n_steps` steps on the usual
windows of L characters, with `train(model, steps=n_steps, warmup=20)`.

In [ ]:
def drope(n_steps=300):
    """The trained RoPE model with its positional encoding dropped, recalibrated for n_steps steps."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
%%time
print(f"rope with its PE dropped, no recalibration: {loss_by_position(drope(0), L).mean():.3f} at L")
drope_results = extrapolation_experiment({
    "rope":              models["rope"],
    "DroPE (300 steps)": drope(300),
    "none":              models["none"],
}, [L, 2 * L, 4 * L, 8 * L, 16 * L])
show_results(drope_results, "DroPE: drop RoPE, recalibrate for 300 steps")

**Question B1.** Compare the DroPE model with the RoPE model it came from and with `none`, the model trained without
positional encoding from the start. What did 300 steps of recalibration buy, and what did they cost?

*Your answer here*

### B2. NoPE: no positional encoding at all


The NoPE paper reports that, on small algorithmic tasks, a decoder without any positional encoding generalises to longer
inputs better than models with explicit encodings. Here is how much each of our models loses between L and 8 L:

In [ ]:
print("validation loss at L, at 8 L, and the difference")
for name in ["alibi", "rope", "none", "sinusoidal"]:
    at_L, at_8L = results[name][L], results[name][8 * L]
    print(f"  {name:11s} {at_L:.3f}   {at_8L:.3f}   {at_8L - at_L:+.3f}")

**Question B2.** Does `none` generalise to longer contexts better than the explicit encodings on our task? Use
Exercise 1 to explain what kind of position information a model without positional encoding can build for itself.

*Your answer here*